<a href="https://colab.research.google.com/github/CodeOfDuty007/Shapley_reg_PhotoAnalysis/blob/main/PostProcessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import csv
import numpy as np
from pathlib import Path
from collections import defaultdict

# ── Config (edit these paths) ──────────────────────────────────
INPUT_PATH  = "final_results.json"   # change if needed
OUTPUT_PATH = "pipeline_output.csv"
TOP_N_SHAP  = 15
# ───────────────────────────────────────────────────────────────

# 1. Load
with open(INPUT_PATH, "r") as f:
    data = json.load(f)
print(f"[1/7] Loaded {len(data):,} records")

# 2. Extract & validate
records, missing = [], 0
for d in data:
    rf, sc, shv = d.get("raw_features",{}), d.get("scores",{}), d.get("shap_values",{})
    sat, con = rf.get("Saturation_Proxy"), rf.get("Contrast_Proxy")
    if sat is None or con is None:
        missing += 1; continue
    records.append({
        "photo_id"        : d.get("photo_id",""),
        "photo_url"       : d.get("photo_url",""),
        "saturation_proxy": float(sat),
        "contrast_proxy"  : float(con),
        "exposure_time"   : rf.get("ExposureTime_Num"),
        "fnumber"         : rf.get("FNumber"),
        "iso"             : rf.get("ISO"),
        "focal_length"    : rf.get("FocalLength"),
        "actual_score"    : float(sc.get("actual_proxy_score", 0)),
        "predicted_score" : float(sc.get("predicted_score", 0)),
        "shap_saturation" : float(shv.get("Saturation_Proxy", 0)),
        "shap_contrast"   : float(shv.get("Contrast_Proxy", 0)),
    })
print(f"[2/7] Clean: {len(records):,}  |  Dropped: {missing}")

# 3. Descriptive stats
def stats(arr):
    a = np.array([x for x in arr if x is not None], dtype=np.float64)
    return len(a), np.mean(a), np.median(a), np.std(a), np.min(a), np.max(a)

print(f"\n[3/7] Descriptive Statistics")
print(f"{'Field':<22} {'N':>6} {'Mean':>8} {'Median':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("─"*72)
for name, vals in [
    ("Saturation proxy",  [r["saturation_proxy"]  for r in records]),
    ("Contrast proxy",    [r["contrast_proxy"]     for r in records]),
    ("Actual score",      [r["actual_score"]       for r in records]),
    ("Predicted score",   [r["predicted_score"]    for r in records]),
    ("FNumber",           [r["fnumber"]            for r in records]),
    ("ISO",               [r["iso"]                for r in records]),
]:
    n,mn,med,sd,mi,mx = stats(vals)
    print(f"  {name:<20} {n:>6,} {mn:>8.3f} {med:>8.3f} {sd:>8.3f} {mi:>8.3f} {mx:>8.3f}")

# 4. Correlations
sat    = np.array([r["saturation_proxy"]  for r in records])
con    = np.array([r["contrast_proxy"]    for r in records])
actual = np.array([r["actual_score"]      for r in records])
pred   = np.array([r["predicted_score"]   for r in records])

print(f"\n[4/7] Key Correlations")
print(f"  Saturation → actual score : r = {np.corrcoef(sat, actual)[0,1]:+.3f}")
print(f"  Contrast   → actual score : r = {np.corrcoef(con, actual)[0,1]:+.3f}")
print(f"  Saturation ↔ Contrast     : r = {np.corrcoef(sat, con)[0,1]:+.3f}")
print(f"  Actual ↔ Predicted        : r = {np.corrcoef(actual, pred)[0,1]:+.3f}")

# 5. SHAP importance
totals = defaultdict(float)
for d in data:
    for k,v in d.get("shap_values",{}).items():
        totals[k] += abs(float(v))
ranked = sorted(totals.items(), key=lambda x: x[1], reverse=True)
n = len(data)
print(f"\n[5/7] Top {TOP_N_SHAP} Features by Mean |SHAP|")
max_v = ranked[0][1] / n
for i,(k,total) in enumerate(ranked[:TOP_N_SHAP], 1):
    mv = total/n
    bar = "█" * int((mv/max_v)*30)
    tag = " ← YOUR PROXY" if k in ("Saturation_Proxy","Contrast_Proxy") else ""
    print(f"  {i:>2}. {k:<38} {mv:.5f}  {bar}{tag}")

# 6. Residuals
resid = actual - pred
print(f"\n[6/7] Residual Analysis")
print(f"  MAE  : {np.mean(np.abs(resid)):.4f}")
print(f"  RMSE : {np.sqrt(np.mean(resid**2)):.4f}")
print(f"  Bias : {np.mean(resid):+.4f}")

# 7. Export CSV
for r in records:
    r["residual"] = round(r["actual_score"] - r["predicted_score"], 6)
fieldnames = ["photo_id","photo_url","saturation_proxy","contrast_proxy",
              "exposure_time","fnumber","iso","focal_length",
              "actual_score","predicted_score","shap_saturation","shap_contrast","residual"]
with open(OUTPUT_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(records)
print(f"\n[7/7] Saved → {OUTPUT_PATH}")

# Download
from google.colab import files
files.download(OUTPUT_PATH)

[1/7] Loaded 24,996 records
[2/7] Clean: 24,996  |  Dropped: 0

[3/7] Descriptive Statistics
Field                       N     Mean   Median      Std      Min      Max
────────────────────────────────────────────────────────────────────────
  Saturation proxy     24,996    0.339    0.306    0.200    0.000    0.999
  Contrast proxy       24,996    0.209    0.211    0.070    0.014    0.465
  Actual score         24,996    8.620    8.380    1.345    4.820   15.287
  Predicted score      24,996    8.619    8.629    0.376    6.929   12.397
  FNumber              21,306    4.943    4.000    3.830    0.000  100.000
  ISO                  21,729  561.263  200.000 1378.889    0.000 51200.000

[4/7] Key Correlations
  Saturation → actual score : r = +0.077
  Contrast   → actual score : r = -0.069
  Saturation ↔ Contrast     : r = -0.218
  Actual ↔ Predicted        : r = +0.530

[5/7] Top 15 Features by Mean |SHAP|
   1. FNumber                                0.15246  ████████████████████████████

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Push to GitHub

To push the generated `pipeline_output.csv` file to your GitHub repository, you'll need to follow these steps:

1.  **Generate a GitHub Personal Access Token (PAT)**: If you don't have one, go to your GitHub settings -> Developer settings -> Personal access tokens -> Tokens (classic) -> Generate new token. Give it a descriptive name (e.g., "Colab access") and grant it `repo` scope permissions.
2.  **Store the PAT in Colab Secrets**: In the left-hand sidebar of Colab, click on the "🔑 Secrets" icon. Add a new secret named `GITHUB_TOKEN` and paste your PAT as its value. Make sure to enable "Notebook access" for this secret.
3.  **Define your GitHub Username** in the next cell.